## Processing the cohort into one big table for analysis
Once the data is generated into the bq_exports folder, this script will combine those into one big dataframe to make analysis a LOT easier.

In [13]:
list_of_packages <- c("tidyverse", "purrr")
new_packages <- list_of_packages[!(list_of_packages %in% installed.packages()[, "Package"])]
if (length(new_packages) > 0) {
  # NOTE: this notebook must be run with the "R (conda env:jupyter, writable)"
  # kernel -- the base "R" kernel's package library is read-only, which is why
  # install.packages() used to fail here. See the "How to use Anaconda in the
  # Researcher Workbench" notebook for details.
  install.packages(new_packages)
}
library(tidyverse)


In [14]:
base_dir <- file.path(Sys.getenv("HOME"), "workspace", "hht-epi-bucket", "bq_exports")

dataset_names <- c(
  "zipcodeSocioeconomic",
  "procedureOccurrence",
  "person",
  "ingredientOccurrence",
  "conditionOccurrence"
)

# Find all CSV shard files for a given dataset name, across whatever date
# subfolder currently exists.
find_dataset_files <- function(dataset, base_dir) {
  pattern_dir <- file.path(base_dir, dataset)
  files <- list.files(
    path = pattern_dir,
    pattern = "\\.csv(\\.gz)?$",
    recursive = TRUE,
    full.names = TRUE
  )
  sort(files)
}

file_map <- setNames(lapply(dataset_names, find_dataset_files, base_dir = base_dir), dataset_names)

# Sanity check: show what was found for each dataset before loading anything
for (nm in dataset_names) {
  cat("\n---", nm, "---\n")
  if (length(file_map[[nm]]) == 0) {
    cat("  ** NO FILES FOUND ** check base_dir / folder naming\n")
  } else {
    print(file_map[[nm]])
  }
}


--- zipcodeSocioeconomic ---
[1] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/zipcodeSocioeconomic/20260818/zipcodeSocioeconomic/zipcodeSocioeconomic_000000000000.csv"
[2] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/zipcodeSocioeconomic/20260820/zipcodeSocioeconomic/zipcodeSocioeconomic_000000000000.csv"

--- procedureOccurrence ---
[1] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/procedureOccurrence/20260818/procedureOccurrence/procedureOccurrence_000000000000.csv"
[2] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/procedureOccurrence/20260820/procedureOccurrence/procedureOccurrence_000000000000.csv"

--- person ---
[1] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/person/20260818/person/person_000000000000.csv"
[2] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/person/20260820/person/person_000000000000.csv"

--- ingredientOccurrence ---
[1] "/home/jupyter/workspace/hht-epi-bucket/bq_exports/ingredientOccurrence/20260818/ingredientOccurrence/ingredien

In [15]:
load_dataset <- function(files) {
  if (length(files) == 0) return(NULL)
  purrr::map_dfr(files, ~ readr::read_csv(.x, guess_max = 100000, show_col_types = FALSE))
}

raw_data <- setNames(lapply(dataset_names, function(nm) load_dataset(file_map[[nm]])), dataset_names)

# Quick row/col counts for everything that loaded
purrr::iwalk(raw_data, function(df, nm) {
  if (is.null(df)) {
    cat(nm, ": not loaded (no files found)\n")
  } else {
    cat(nm, ": ", nrow(df), "rows x", ncol(df), "columns\n")
  }
})

zipcodeSocioeconomic :  2416 rows x 13 columns
procedureOccurrence :  1066 rows x 18 columns
person :  2528 rows x 12 columns
ingredientOccurrence :  12140 rows x 26 columns
conditionOccurrence :  52184 rows x 19 columns


In [16]:
purrr::iwalk(raw_data, function(df, nm) {
  cat("\n============================\n")
  cat(nm, "\n")
  cat("============================\n")
  if (!is.null(df)) print(names(df))
})


zipcodeSocioeconomic 
 [1] "american_community_survey_year" "assisted_income"               
 [3] "deprivation_index"              "high_school_education"         
 [5] "id"                             "median_income"                 
 [7] "no_health_insurance"            "observation_datetime_str"      
 [9] "observation_datetime_utc"       "person_id"                     
[11] "poverty"                        "vacant_housing"                
[13] "zip_code"                      

procedureOccurrence 
 [1] "modifier_concept_id"                "T_DISP_modifier_concept_name"      
 [3] "modifier_source_value"              "person_id"                         
 [5] "procedure_concept_id"               "procedure_datetime"                
 [7] "procedure_source_concept_id"        "procedure_source_value"            
 [9] "procedure_type_concept_id"          "T_DISP_procedure_type_concept_name"
[11] "quantity"                           "T_DISP_source_concept_code"        
[13] "T_DISP_sour

In [17]:
# Optional: deeper look at one table at a time, e.g. person
glimpse(raw_data$person)

Rows: 2,528
Columns: 12
$ birth_datetime                    <chr> "1941-06-15 00:00:00 UTC", "1996-06-…
$ ethnicity_concept_id              <dbl> 38003564, 903096, 903096, 38003564, …
$ T_DISP_ethnicity                  <chr> "Not Hispanic or Latino", "PMI: Skip…
$ gender_concept_id                 <dbl> 45878463, 903096, 903096, 45880669, …
$ T_DISP_gender                     <chr> "Female", "PMI: Skip", "PMI: Skip", …
$ person_id                         <dbl> 2175828, 2304816, 7205750, 2064703, …
$ race_concept_id                   <dbl> 2000000008, 903096, 903096, 8516, 85…
$ T_DISP_race                       <chr> "More than one population", "PMI: Sk…
$ self_reported_category_concept_id <dbl> 2000000008, 903096, 903096, 8516, 85…
$ T_DISP_self_reported_category     <chr> "More than one population", "PMI: Sk…
$ sex_at_birth_concept_id           <dbl> 0, 903096, 903096, 903096, 903096, 9…
$ T_DISP_sex_at_birth               <chr> "No matching concept", "PMI: Skip", …


In [18]:
glimpse(raw_data$conditionOccurrence)

Rows: 52,184
Columns: 19
$ condition_concept_id                 <dbl> 4218100, 4218100, 4218100, 421810…
$ condition_end_datetime               <chr> NA, NA, NA, NA, NA, NA, NA, "2020…
$ condition_source_concept_id          <dbl> 35206733, 35206733, 35206733, 352…
$ condition_source_value               <chr> "D59.2", "D59.2", "D59.2", "D59.2…
$ condition_start_datetime             <chr> "2013-04-24 12:55:24 UTC", "2013-…
$ condition_status_concept_id          <dbl> NA, NA, NA, NA, NA, NA, NA, 0, 0,…
$ T_DISP_condition_status_concept_name <chr> NA, NA, NA, NA, NA, NA, NA, "No m…
$ condition_status_source_value        <chr> NA, NA, NA, NA, NA, NA, NA, "No m…
$ condition_type_concept_id            <dbl> 32020, 32020, 32020, 32020, 32020…
$ T_DISP_condition_type_concept_name   <chr> "EHR encounter diagnosis", "EHR e…
$ person_id                            <dbl> 1407663, 1407663, 1407663, 140766…
$ T_DISP_source_concept_code           <chr> "D59.2", "D59.2", "D59.2", "D59.2…
$ T_DISP_source

In [19]:
glimpse(raw_data$procedureOccurrence)

Rows: 1,066
Columns: 18
$ modifier_concept_id                <dbl> 0, 0, 0, NA, NA, 0, 0, 0, NA, NA, 0…
$ T_DISP_modifier_concept_name       <chr> "No matching concept", "No matching…
$ modifier_source_value              <chr> "No matching concept", "No matching…
$ person_id                          <dbl> 7784701, 1049947, 1898027, 1982033,…
$ procedure_concept_id               <dbl> 40485932, 40485932, 40485932, 40485…
$ procedure_datetime                 <chr> "2012-03-29 11:25:00 UTC", "2012-11…
$ procedure_source_concept_id        <dbl> 2002688, 2002688, 2002688, 2002688,…
$ procedure_source_value             <chr> "45.16", "45.16", "45.16", "45.16",…
$ procedure_type_concept_id          <dbl> 38000252, 44786631, 44786631, 32817…
$ T_DISP_procedure_type_concept_name <chr> "Inpatient header - 2nd position", …
$ quantity                           <dbl> 1, NA, NA, 1, NA, NA, NA, NA, NA, N…
$ T_DISP_source_concept_code         <chr> "45.16", "45.16", "45.16", "45.16",…
$ T_DISP_source_

In [20]:
glimpse(raw_data$ingredientOccurrence)

Rows: 12,140
Columns: 26
$ days_supply                   <dbl> 90, 84, 90, 90, 90, 90, 90, 90, 90, NA, …
$ dose_unit_source_value        <lgl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …
$ drug_concept_id               <dbl> 42902176, 42902176, 42902176, 42902176, …
$ drug_exposure_end_datetime    <chr> "2018-02-11 12:07:07 UTC", "2018-11-07 0…
$ drug_exposure_start_datetime  <chr> "2017-11-13 12:07:07 UTC", "2018-08-15 0…
$ drug_source_concept_id        <dbl> 4317453, 4317453, 4317453, 4317453, 4317…
$ drug_source_value             <chr> "4025177", "4025177", "4025177", "402517…
$ drug_type_concept_id          <dbl> 38000175, 38000175, 38000175, 38000175, …
$ T_DISP_drug_type_concept_name <chr> "Prescription dispensed in pharmacy", "P…
$ lot_number                    <lgl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …
$ person_id                     <dbl> 8385580, 8385580, 8385580, 8385580, 8385…
$ quantity                      <dbl> 3, 3, 3, 3, 3, 3, 3, 3, 3, NA, NA, 1, 1,…
$ refills      

In [21]:
glimpse(raw_data$zipcodeSocioeconomic)

Rows: 2,416
Columns: 13
$ american_community_survey_year <dbl> 2017, 2017, 2017, 2017, 2017, 2017, 201…
$ assisted_income                <dbl> 5.165908, 5.165908, 5.165908, 5.165908,…
$ deprivation_index              <dbl> 0.1625441, 0.1625441, 0.1625441, 0.1625…
$ high_school_education          <dbl> 96.18835, 96.18835, 96.18835, 96.18835,…
$ id                             <dbl> 52794, 431517, 608707, 704156, 685135, …
$ median_income                  <dbl> 126454.49, 126454.49, 126454.49, 126454…
$ no_health_insurance            <dbl> 2.032504, 2.032504, 2.032504, 2.032504,…
$ observation_datetime_str       <dttm> 2019-03-19 16:21:22, 2022-09-09 14:36:…
$ observation_datetime_utc       <chr> "2019-03-19 16:21:22 UTC", "2022-09-09 …
$ person_id                      <dbl> 2495533, 9194415, 8129242, 3446758, 232…
$ poverty                        <dbl> 6.550429, 6.550429, 6.550429, 6.550429,…
$ vacant_housing                 <dbl> 5.271112, 5.271112, 5.271112, 5.271112,…
$ zip_code      

In [22]:
## TODO: select columns to drop
columns_to_drop <- list(
  person               = character(0),  # e.g. c("location_id", "provider_id")
  conditionOccurrence  = character(0),  # e.g. c("condition_type_concept_id")
  procedureOccurrence  = character(0),
  ingredientOccurrence = character(0),
  zipcodeSocioeconomic = character(0)
)

cleaned_data <- purrr::imap(raw_data, function(df, nm) {
  if (is.null(df)) return(NULL)
  drop_cols <- columns_to_drop[[nm]]
  if (is.null(drop_cols)) drop_cols <- character(0)
  df %>% select(-any_of(drop_cols))
})

# Confirm the trim worked
purrr::iwalk(cleaned_data, function(df, nm) {
  if (!is.null(df)) cat(nm, ": ", ncol(df), "columns remaining\n")
})

zipcodeSocioeconomic :  13 columns remaining
procedureOccurrence :  18 columns remaining
person :  12 columns remaining
ingredientOccurrence :  26 columns remaining
conditionOccurrence :  19 columns remaining


Next, we generate the new cleaned table

In [23]:
cleaned_dir <- file.path(base_dir, "cleaned")
dir.create(cleaned_dir, showWarnings = FALSE)

purrr::iwalk(cleaned_data, function(df, nm) {
  if (!is.null(df)) {
    saveRDS(df, file.path(cleaned_dir, paste0(nm, ".rds")))
    cat("Saved:", file.path(cleaned_dir, paste0(nm, ".rds")), "\n")
  }
})

Saved: /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/zipcodeSocioeconomic.rds 
Saved: /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/procedureOccurrence.rds 
Saved: /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/person.rds 
Saved: /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/ingredientOccurrence.rds 
Saved: /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/conditionOccurrence.rds 


## Merge all cleaned tables into one big dataframe for analysis

`person` and `zipcodeSocioeconomic` are **person-level** (one row per person, or per person per SDOH survey). `conditionOccurrence`, `procedureOccurrence`, and `ingredientOccurrence` are **event-level** (many rows per person).

To combine tables of different grains into a single dataframe, we stack the three event-level tables into one tidy "long" table (one row per clinical event, tagged with its domain: condition / procedure / drug), then left-join person demographics and zip-code socioeconomic/geographic data onto every event row. The result, `analytic_df`, is one big dataframe that supports all of the planned analyses from a single table:

- **Prevalence / incidence**: filter `domain` and `concept_name`, then count distinct `person_id` (prevalence) or take `min(event_date)` per person (incidence)
- **Geographic distribution**: `zip_code` (plus the socioeconomic/deprivation columns)
- **Symptom/complication rates** (e.g. anemia, epistaxis, AVMs): `domain == "condition"`
- **Medication use** (e.g. bevacizumab, pomalidomide): `domain == "drug"`
- **Procedures** (e.g. control of epistaxis): `domain == "procedure"`


In [24]:
# Standardize each event-level table to a common set of columns so they can
# be stacked with bind_rows(): person_id, domain, concept_name,
# source_vocabulary / source_concept_code, and event_date.
standardize_events <- function(df, domain_label, date_col) {
  if (is.null(df)) return(NULL)
  df %>%
    transmute(
      person_id,
      domain = domain_label,
      concept_name = .data[["T_DISP_standard_concept_name"]],
      source_vocabulary = .data[["T_DISP_standard_vocabulary"]],
      source_concept_code = as.character(.data[["T_DISP_standard_concept_code"]]),
      event_date = as.Date(.data[[date_col]])
    )
}

events_list <- list(
  standardize_events(cleaned_data$conditionOccurrence,  "condition", "condition_start_datetime"),
  standardize_events(cleaned_data$procedureOccurrence,  "procedure", "procedure_datetime"),
  standardize_events(cleaned_data$ingredientOccurrence, "drug",      "drug_exposure_start_datetime")
)
all_events <- bind_rows(events_list)

# Person-level demographics
person_df <- cleaned_data$person

# Zip-code / socioeconomic data can have multiple survey rows per person
# (american_community_survey_year); keep the most recent one per person.
zip_df <- cleaned_data$zipcodeSocioeconomic
if (!is.null(zip_df)) {
  zip_df <- zip_df %>%
    group_by(person_id) %>%
    arrange(desc(observation_datetime_utc), .by_group = TRUE) %>%
    slice(1) %>%
    ungroup()
}

# Join events -> person demographics -> zip/SES/geography into one big
# analysis-ready dataframe.
analytic_df <- all_events %>%
  left_join(person_df, by = "person_id")
if (!is.null(zip_df)) {
  analytic_df <- analytic_df %>% left_join(zip_df, by = "person_id")
}

cat("analytic_df:", nrow(analytic_df), "rows x", ncol(analytic_df), "columns\n")
cat("Unique persons in analytic_df:", n_distinct(analytic_df$person_id),
    "out of", n_distinct(person_df$person_id), "in the person table\n")
glimpse(analytic_df)


Warning message in left_join(., person_df, by = "person_id"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 8 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


analytic_df: 130780 rows x 29 columns
Unique persons in analytic_df: 602 out of 1264 in the person table
Rows: 130,780
Columns: 29
$ person_id                         <dbl> 1407663, 1407663, 1407663, 1407663, …
$ domain                            <chr> "condition", "condition", "condition…
$ concept_name                      <chr> "Hemolytic anemia caused by drugs", …
$ source_vocabulary                 <chr> "SNOMED", "SNOMED", "SNOMED", "SNOME…
$ source_concept_code               <chr> "81711008", "81711008", "81711008", …
$ event_date                        <date> 2013-04-24, 2013-04-24, 2013-05-06,…
$ birth_datetime                    <chr> "1957-06-15 00:00:00 UTC", "1957-06-…
$ ethnicity_concept_id              <dbl> 38003563, 38003563, 38003563, 380035…
$ T_DISP_ethnicity                  <chr> "Hispanic or Latino", "Hispanic or L…
$ gender_concept_id                 <dbl> 45878463, 45878463, 45878463, 458784…
$ T_DISP_gender                     <chr> "Female", "Female", "Female

In [25]:
# Save the combined analysis dataframe alongside the per-domain cleaned tables
saveRDS(analytic_df, file.path(cleaned_dir, "analytic_df.rds"))
write_csv(analytic_df, file.path(cleaned_dir, "analytic_df.csv"))
cat("Saved combined analysis dataframe to:\n ",
    file.path(cleaned_dir, "analytic_df.rds"), "\n ",
    file.path(cleaned_dir, "analytic_df.csv"), "\n")

# Quick sanity check: event counts by domain and concept, useful for spotting
# whether a concept set in Get_cohort.ipynb returned zero/unexpectedly few rows.
analytic_df %>%
  count(domain, concept_name, sort = TRUE) %>%
  print(n = 40)


Saved combined analysis dataframe to:
  /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/analytic_df.rds 
  /home/jupyter/workspace/hht-epi-bucket/bq_exports/cleaned/analytic_df.csv 
# A tibble: 201 × 3
   domain    concept_name                                            n
   <chr>     <chr>                                               <int>
 1 condition Osler hemorrhagic telangiectasia syndrome           28764
 2 condition Anemia                                              12924
 3 condition Iron deficiency anemia                              10192
 4 condition Iron deficiency anemia due to blood loss            10136
 5 condition Hemoglobin SS disease with crisis                    9652
 6 condition Bleeding from nose                                   8780
 7 drug      folic acid                                           4696
 8 condition Iron deficiency                                      4232
 9 drug      folic acid 1 MG Oral Tablet                          3476
10 cond

### Next steps

`analytic_df` (and the per-domain `cleaned/*.rds` files) are ready for the epidemiology analysis: prevalence/incidence of HHT, geographic distribution, symptom/complication rates, medication use, and procedures. If a variable of interest is missing from the counts above (e.g. a medication or condition concept), update the concept set(s) in `Get_cohort.ipynb`, re-run it to refresh `bq_exports/`, and re-run this notebook.
